# 1 · A contact card

**The problem.** You are writing a contacts service. A contact is a handful of text fields today, but the same data
has to be read by a TypeScript front end, stored as JSON, edited by hand as YAML, and checked before it is trusted.
In plain Python you would reach for a `dataclass`. Each of those needs would then add a separate description of the
same record: a JSON Schema, a TypeScript interface, a validation layer. Sooner or later the copies disagree.

This framework takes a different route. **The description of the data is itself data.** You build a *schema* once, at
runtime, with a small fluent DSL. Everything else works from that one schema value: instances, serialization,
validation, and in time generated code for other languages.

This first case study builds a single record and covers the handful of ideas every later one relies on:

1. Schemas are values, built with *builders*.
2. Instances are built with builders too. They are read-only once built.
3. A builder ends in one of `create()`, `clone()` or `update()`.
4. A *Spec* is either a value or a function that configures a builder.
5. Every property is optional. "Not set" is a real state, distinct from `""`, `0` or `False`.

In [1]:
from mbse_schemas.Framework import Proxies, Schemas

## Describing the record

A schema for an object with properties is an `OfObject`. You get one from its builder:
`Schemas.OfObject.Builder()`. Each property is described by a function that receives a *property builder*
(`prop`), names the property and gives its type. The type is described by a function too: it receives an "any
schema" builder (`t`) and picks a kind. Here every kind is `as_native(...)`: a value native to the programming
language (`str`, `int`, `float`, `bool`, `bytes`).

Read the chain from left to right. It says what it builds, and the final `.create()` hands you the finished schema.

In [2]:
Contact = (
    Schemas.OfObject.Builder()
    .properties(
        lambda prop: prop.name('given_name').of(lambda t: t.as_native(str)),
        lambda prop: prop.name('family_name').of(lambda t: t.as_native(str)),
        lambda prop: prop.name('birth_date').of(lambda t: t.as_native(str)),  # ISO 8601, e.g. '1852-05-04'
        lambda prop: prop.name('vip').of(lambda t: t.as_native(bool)),
        lambda prop: prop.name('photo').of(lambda t: t.as_native(bytes)),
    )
    .create()
)
Contact

_ObjectData(properties={'given_name': _NativeData(type=<class 'str'>), 'family_name': _NativeData(type=<class 'str'>), 'birth_date': _NativeData(type=<class 'str'>), 'vip': _NativeData(type=<class 'bool'>), 'photo': _NativeData(type=<class 'bytes'>)}, adjacencies={}, singleton=None)

### Why lambdas instead of a class?

A Python class is fine for Python, but other programs can't read it. The schema above is a plain data structure.
It can be inspected, compared, printed and (once meta-schemas land) serialized, and code for other languages can be
generated from it.

The lambdas are only a way to *configure* a builder in place. They don't end up in the schema. Each lambda
receives a builder, calls fluent methods on it, and returns it. This "function that configures a builder" is called
a **Spec**, and it appears everywhere in the framework.

A Spec can also be a finished value. `as_native(str)` passes the type directly. An existing schema can be passed
wherever a Spec is expected, and it is reused, not copied:

In [3]:
Text = Schemas.OfNative.Data(str)

direct = Schemas.OfAny.Builder().as_native(str).create()
configured = Schemas.OfAny.Builder().as_native(lambda n: n.type(str)).create()
assert direct == configured == Text

Named = (
    Schemas.OfObject.Builder()
    .properties(
        lambda prop: prop.name('first').of(Text),  # a value Spec: an existing schema, reused
        lambda prop: prop.name('last').of(lambda t: t.as_native(str)),  # a callable Spec: built in place
    )
    .create()
)
assert Named.properties['first'] is Text and Named.properties['last'] == Text

Schema equality is structural (`==` compares what the schemas say). Use `is` when you mean "the very same schema
object".

## Checking the schema, when you ask

Building never validates. That lets you build schemas in steps: case study 2 needs a moment when two schemas refer to
each other and neither is complete yet. When you want to know whether a schema makes sense, ask it:

In [4]:
assert Contact.validate() == []

Broken = Schemas.OfObject.Builder().properties(lambda prop: prop.name('age').of(lambda t: t.as_native(complex))).create()
Broken.validate()

["property 'age': unsupported native type <class 'complex'>"]

`validate()` returns a list of problems rather than raising, so a tool can report all of them at once.

## Making contacts

To make instances, register the schema under a global name. The name is how other programs, and serialized data, will
refer to this kind of object:

In [5]:
Proxies.register('Contact', Contact)
Builders = Proxies.Builders

`Builders.Contact()` returns an *instance builder*. It has one fluent setter per property, generated from the schema.
That is the "dynamic" implementation, *proxies*: no code generation step, and the schema drives everything at
runtime.

In [6]:
alice = (
    Builders.Contact()
    .given_name('Alice')
    .family_name('Liddell')
    .birth_date('1852-05-04')
    .vip(False)
    .create()
)
print(alice.given_name, alice.family_name, alice.birth_date, alice.vip)

Alice Liddell 1852-05-04 False


## Instances are read-only

Properties read like attributes, but you can't assign to them:

In [7]:
try:
    alice.given_name = 'Alicia'
except AttributeError as error:
    print('AttributeError:', error)

AttributeError: proxy properties are read-only; use a builder


**Why?** An object here is more than its fields. In case study 2 it gains *relations* to other objects, and a change
can touch several objects at once. Routing every change through a builder gives one place where a change is
assembled and one moment when it takes effect: `update()`. It also keeps the door open for recording changes as data
(*mutations*) later, without changing how you write code.

## Changing things: `update()` and `clone()`

Pass an existing instance to the builder. The builder starts from a copy of that instance's state. Then finish with:

- `update()` to write the changes back into that same instance, or
- `clone()` to produce a new instance and leave the original untouched.

`create()` is only for builders that started empty. Each ending is valid in exactly one situation, so a builder
can never quietly do the wrong one.

In [8]:
same = Builders.Contact(alice).family_name('Hargreaves').update()
assert same is alice and alice.family_name == 'Hargreaves'

sister = Builders.Contact(alice).given_name('Lorina').clone()
assert sister is not alice
print(sister.given_name, sister.family_name, '| original:', alice.given_name)

Lorina Hargreaves | original: Alice


In [9]:
for wrong in (lambda: Builders.Contact(alice).create(), lambda: Builders.Contact().update()):
    try:
        wrong()
    except ValueError as error:
        print('ValueError:', error)

ValueError: create() is only valid without a source instance; use clone() or update()
ValueError: update() is only valid with a source instance


## "Not set" is a state of its own

Every property is optional: nothing is mandatory unless a constraint says so. A property that was never set is
*absent*, and reading it raises. It doesn't quietly return `None`:

In [10]:
try:
    alice.photo
except AttributeError as error:
    print('AttributeError:', error)

AttributeError: property 'photo' is not set


**Why not `None`?** Because `None` would be a value, and then "the photo is None" and "nobody said anything about
the photo" would look the same. They mean different things. In a partial update, for example, one means "remove
it" and the other means "leave it alone". Likewise `vip(False)` is a *present* `False`, not an absent flag.

To test for presence without raising, use `getattr` with a default, or `hasattr`:

In [11]:
print(hasattr(alice, 'vip'), hasattr(alice, 'photo'), getattr(alice, 'photo', 'no photo'))

True False no photo


To make a property absent again, *clear* it. This is the other form of Spec, a function. Every setter accepts
either the value or a function that receives a *value handle* `v`. `v.set(...)` and `v.clear()` are the two things
you can do to it. So `.vip(True)` is shorthand for `.vip(lambda v: v.set(True))`:

In [12]:
Builders.Contact(alice).vip(lambda v: v.set(True)).birth_date(lambda v: v.clear()).update()
print(alice.vip, hasattr(alice, 'birth_date'))

True False


## Builders don't check types

The instance builder accepts any value for any property. That's deliberate, and it's the same rule as for schemas:
building is cheap and permissive, and checking is something you ask for. Here the photo is set to a string where
the schema says `bytes`:

In [13]:
sloppy = Builders.Contact().given_name('Bob').photo('not bytes').create()
print(repr(sloppy.photo))

'not bytes'


Nothing complained yet. Case study 5 shows the two places that do: the validator, which reports every problem with
a path, and the serializers, which refuse to write data that doesn't match its schema.

## What you learned

- A schema is a value built with `Schemas.OfObject.Builder()...create()`. `validate()` checks it when you ask.
- A Spec is a value, or a function that configures a builder and returns it.
- `Proxies.register(name, schema)` makes `Proxies.Builders.<name>()` available. Instances are read-only.
- Builders end with `create()` (from nothing), `update()` (in place) or `clone()` (a copy).
- Properties are optional. Absent isn't `None`, and `v.clear()` makes a property absent.

**Next:** a contact rarely has just one address. Case study 2 introduces *relations*, the framework's only collection
type, and explains why there are no lists.